Setup and load

In [1]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error, mean_squared_error

RANDOM_STATE = 42

clf_df = pd.read_csv("data/classification_escalation.csv")
reg_df = pd.read_csv("data/resolution_time_regression.csv")
display(clf_df.head(), reg_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [2]:
print("Classification:", clf_df.shape, "| Regression:", reg_df.shape)
display(clf_df.isna().sum().to_frame("missing (clf)"), reg_df.isna().sum().to_frame("missing (reg)"))
print(clf_df["escalated"].value_counts().to_dict(), "| prevalence:", clf_df["escalated"].mean().round(4))

Classification: (900, 7) | Regression: (800, 7)


,missing (clf)
ticket_id,0
complexity_score,22
team_experience_years,36
priority,13
channel,9
customer_tenure_months,31
escalated,0


,missing (reg)
ticket_id,0
complexity_score,16
team_experience_years,24
priority,0
channel,0
customer_tenure_months,20
resolution_time_hours,0


{0: 709, 1: 191} | prevalence: 0.2122


## Step 1 – Data overview
- Two datasets: classification (900 tickets, target `escalated`) and regression (800 tickets, target `resolution_time_hours`). Both share the same 5 predictors; `ticket_id` is an identifier and is excluded.
- Missing values appear in the numeric predictors of both files, and also in `priority` and `channel` of the classification file, so imputation is required.
- About 21% of tickets are escalated: the classes are imbalanced, which is why F1 (not accuracy) is the primary metric.

In [3]:
num_cols = ["complexity_score", "team_experience_years", "customer_tenure_months"]
cat_cols = ["priority", "channel"]

def make_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ])

X = clf_df.drop(columns=["ticket_id", "escalated"])
y = clf_df["escalated"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

print(X_train.shape, X_test.shape)
print("Positives  train:", y_train.sum(), round(y_train.mean(), 4), "| test:", y_test.sum(), round(y_test.mean(), 4))

(675, 5) (225, 5)
Positives  train: 143 0.2119 | test: 48 0.2133


## Step 2 – Experiment A: split and preprocessing
- `ticket_id` is dropped (an identifier, not a predictor). Target = `escalated`; positive class = 1.
- 75/25 **stratified** split: the escalation rate is 21.2% in train and 21.3% in test, so evaluation is on a representative test set with 48 positives. The test set is then locked away.
- Numeric: median imputation (robust to outliers) + standardization (needed by k-NN and helpful for logistic regression). Categorical: most-frequent imputation + one-hot (`handle_unknown="ignore"` so unseen categories don't crash).
- Preprocessing is built by `make_preprocessor()` and placed **inside every Pipeline**, so it is re-fit on the training part of each CV fold only (no leakage).